# Export CSVs

To be imported in `./statistics.Rmd` for statistical analysis and `./mTRF_plots.ipynb` for plotting.
- `./results_full.csv`
- `./results_models.csv`

In [10]:
import os, glob, h5py
import numpy as np
import pandas as pd

RES_ATT = '/Users/nadastojanovic/Development/mphil/2_mTRF/4_results/Attended/'
RES_UNATT = '/Users/nadastojanovic/Development/mphil/2_mTRF/4_results/Unattended_with_MuR/'

OUT_DIR = '/Users/nadastojanovic/Development/mphil/3_stats/'

GROUP_LABEL = {'High': 'Active', 'Low': 'Moderate', 'No': 'Inactive'}

PID_TO_AOA = {
    101:4, 102:7, 103:5, 104:0, 105:11, 106:0, 107:7, 108:11,
    109:7, 111:0, 112:4, 113:0, 114:0, 115:6, 117:0, 118:10,
    119:8, 121:2, 122:5, 123:0, 124:5, 125:1, 126:8, 128:1,
    129:11, 130:6, 131:0, 132:6, 133:8, 134:10, 135:9, 136:0,
    137:11, 138:8, 139:4, 140:4, 141:4, 142:10, 143:5, 145:11,
    146:0, 147:6, 148:6, 149:7, 150:5, 151:8, 152:11, 153:0,
}

INDIV_MODELS = {
    'env','phon_onsets','morph','word_onsets','artic',
    'word_freq','pos_seg_freq','bi_freq','phon_surp','phon_ent',
    'word_surp','word_ent','synt_depth','synt_deps'
}

SKIP = {'participant_ID','condition','usage_group','time_lags','feature_bands','full'}

In [11]:
# helper: read matlab char array
def read_str(ds):
    return ''.join(chr(int(c)) for c in np.array(ds).flatten())

# helper: load results files
def load_dir(res_dir, attn_label):
    full_rows, model_rows = [], []

    for fpath in sorted(glob.glob(res_dir + '*.mat')):
        with h5py.File(fpath, 'r') as f:
            TRF = f['TRF']
            pid = int(read_str(TRF['participant_ID']))
            cond = read_str(TRF['condition'])

            meta = dict(
                pid = pid,
                cond = cond,
                attn = attn_label,
                usage_group = GROUP_LABEL[read_str(TRF['usage_group'])],
                aoa = PID_TO_AOA[pid],
                language = cond[:3],
                interference = 'MuR' if cond[-3:] == 'MuR' else 'Lang',
            )

            # df_full - full model results and 
            # null (all test stim columns permuted)
            if 'full' in TRF:
                full_rows.append({**meta,
                    'r_full': float(np.array(TRF['full']['r_full_avg']).squeeze()),
                    'r_null': float(np.array(TRF['full']['r_null_avg']).squeeze())
                })

            # df_models - solo and composite model results and 
            # nulls (corresponding test stim columns permuted)
            for gname in TRF.keys():
                if gname in SKIP: # metadata
                    continue
                    
                grp = TRF[gname]
                is_indiv = gname in INDIV_MODELS # store r_full only for solo models,
                                                 # nan for composites
                model_rows.append({**meta,
                    'model': gname,
                    'r_null_avg': float(np.array(grp['r_null_avg']).squeeze()),
                    'r_corr': float(np.array(grp['r_corr_avg']).squeeze()),
                    'r_full': float(np.array(grp['r_full_avg']).squeeze()) if is_indiv else np.nan,
                })

    return pd.DataFrame(full_rows), pd.DataFrame(model_rows)

In [12]:
full_att, models_att = load_dir(RES_ATT, 'Attended')
full_unatt, models_unatt = load_dir(RES_UNATT, 'Unattended')

df_full = pd.concat([full_att, full_unatt], ignore_index=True)
df_models = pd.concat([models_att, models_unatt], ignore_index=True)

In [13]:
df_full.to_csv(OUT_DIR + 'results_full.csv', index=False)
df_models.to_csv(OUT_DIR + 'results_models.csv', index=False)